# Hail Mary Distilled — Guided Unsloth Colab Notebook

This notebook fine-tunes a small student model on the combined sci-fi reasoning dataset and then compares the base model against the tuned model.

Teaching goals:
- understand why we use a small student model on Colab Free
- see how instruction data becomes chat training text
- run a tiny smoke test before the full run
- compare base vs fine-tuned behavior on a fixed benchmark

## 1) Install dependencies

In [ ]:
!pip -q install unsloth transformers datasets accelerate trl peft bitsandbytes huggingface_hub pandas

## 2) Optional: mount Google Drive

Why this matters:
- Colab sessions can reset.
- Drive helps preserve checkpoints and outputs between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3) Configure model, dataset, and output paths

In [ ]:
from pathlib import Path

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
TRAIN_DATA_PATH = Path('/content/train_v1_combined.jsonl')
EVAL_DATA_PATH = Path('/content/eval_prompts.jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/hail-mary-student-lora')
HUB_MODEL_ID = 'your-username/hail-mary-inspired-student-lora'
HUB_DATASET_ID = 'your-username/hail-mary-inspired-sci-fi-instruct'
RUN_FULL_TRAINING = False
SMOKE_TEST_STEPS = 20

## 4) Upload dataset files

Upload these files from your repo:
- `data/train_v1_combined.jsonl`
- `data/eval_prompts.jsonl`

Simple option:
- upload them into `/content` using Colab Files or Google Drive
- if you store them in Drive, update the paths above before continuing

Teaching note:
- We train on the combined dataset because the seed set alone is too small for a meaningful demo.
- We keep evaluation separate so we can compare behavior consistently.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(sorted(uploaded.keys()))

## 5) Load and inspect the raw data

In [ ]:
import json
import pandas as pd
from datasets import Dataset

train_rows = [json.loads(line) for line in TRAIN_DATA_PATH.read_text().splitlines() if line.strip()]
eval_rows = [json.loads(line) for line in EVAL_DATA_PATH.read_text().splitlines() if line.strip()]

train_df = pd.DataFrame(train_rows)
display(train_df.head(3))
print('training rows:', len(train_rows))
print('eval rows:', len(eval_rows))
print('difficulty counts:')
print(train_df['difficulty'].value_counts())

## 6) Load the base model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

## 7) Format chat examples into training text

Teaching note:
- Chat models learn from serialized conversations.
- The tokenizer's chat template turns structured messages into the exact text pattern the base model expects.

In [ ]:
dataset = Dataset.from_list(train_rows)

def format_row(example):
    messages = [
        {'role': 'system', 'content': example['system']},
        {'role': 'user', 'content': example['user']},
        {'role': 'assistant', 'content': example['assistant']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

dataset = dataset.map(format_row)
print(dataset[0]['text'][:900])

## 8) Run a smoke test first

Why this matters:
- We want to prove the pipeline works before spending time on a longer run.
- This is a core training habit: test the system small, then scale.

In [ ]:
smoke_dataset = dataset.select(range(min(12, len(dataset))))
full_dataset = dataset
train_dataset = full_dataset if RUN_FULL_TRAINING else smoke_dataset
print('using rows:', len(train_dataset))

## 9) Train with `SFTTrainer`

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=SMOKE_TEST_STEPS if not RUN_FULL_TRAINING else -1,
    num_train_epochs=2 if RUN_FULL_TRAINING else 1,
    learning_rate=2e-4,
    logging_steps=1,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    output_dir=str(OUTPUT_DIR),
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

trainer_stats = trainer.train()
trainer_stats

## 10) Define a reusable generation helper

In [ ]:
FastLanguageModel.for_inference(model)
FastLanguageModel.for_inference(base_model)

SYSTEM_PROMPT = 'You are a calm, science-literate assistant helping a space crew solve hard problems with optimism and honesty.'

def generate_response(active_model, user_prompt, max_new_tokens=220):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(active_model.device)
    outputs = active_model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        do_sample=False,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split(user_prompt, 1)[-1].strip() if user_prompt in decoded else decoded.strip()

## 11) Quick single-prompt check

In [ ]:
test_prompt = 'How should a crew talk about uncertainty after detecting a strange repeating signal?'
print('BASE MODEL:\n')
print(generate_response(base_model, test_prompt))
print('\n' + '=' * 80 + '\n')
print('FINE-TUNED MODEL:\n')
print(generate_response(model, test_prompt))

## 12) Base vs fine-tuned benchmark comparison

Teaching note:
- We are not looking for perfect answers.
- We are checking whether the tuned model is more aligned with the project style: clearer, calmer, and better at handling uncertainty.

In [ ]:
comparison_rows = []
for row in eval_rows:
    base_answer = generate_response(base_model, row['user'])
    tuned_answer = generate_response(model, row['user'])
    comparison_rows.append({
        'id': row['id'],
        'category': row['category'],
        'prompt': row['user'],
        'base_answer': base_answer,
        'tuned_answer': tuned_answer,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df[['id', 'category', 'prompt']])

In [ ]:
example_index = 0
row = comparison_rows[example_index]
print('PROMPT:\n')
print(row['prompt'])
print('\n' + '=' * 80 + '\n')
print('BASE ANSWER:\n')
print(row['base_answer'])
print('\n' + '=' * 80 + '\n')
print('TUNED ANSWER:\n')
print(row['tuned_answer'])

## 13) Manual scoring rubric

For each evaluation prompt, score the base and tuned answers from 1 to 5 on:
- instruction following
- tone consistency
- reasoning clarity
- uncertainty handling
- hallucination resistance

The goal is not to make the tuned model win every category. The goal is to learn where the dataset is helping and where it is still weak.

In [ ]:
comparison_df.to_csv('/content/eval_comparison.csv', index=False)
print('saved /content/eval_comparison.csv')

## 14) Save the trained adapter

Save locally first. This is simpler and safer than pushing immediately.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tuned_eval_model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print('saved to', OUTPUT_DIR)

In [ ]:
print('Next step options:')
print('1. Keep the saved folder in Drive and upload later from a clean session.')
print('2. Use the helper scripts in this repo to clean and publish the model and dataset.')
print('3. Update the Hugging Face model card before any re-upload.')

## 15) What to do after this run

- If the fine-tuned model sounds better but still inconsistent, add more high-quality synthetic rows.
- If it overfits the training style, diversify prompts and shorten some answers.
- If the notebook fails on the smoke test, fix that before running the full dataset.
- Once the full run looks good, publish the model card, dataset card, and your LinkedIn write-up.